# Reversible Circuits Project: Phase 3

A heuristic optimizer built from the transformation rules of Feng and Li
(*A Complete Set of Transformation Rules for Reversible Circuits*, IEEE TCAD, 2026).

**How to run:** choose **Runtime > Run all** from the menu. The whole notebook takes under a minute.

| Step | What it does |
|---|---|
| 1 | Creates the Phase 1 and Phase 2 files (unchanged) |
| 2 | Creates `optimizer.py`: cost model, rewrite moves, and the search |
| 3 | Creates `exact.py`: the true minimum circuit of every 3-line function |
| 4 | Creates `verify_phase3.py`, the Phase 3 tests and experiments |
| 5 | Runs them |
| 6 | An example you can edit and re-run |

This notebook is self-contained. You do not need to run the earlier notebooks first.

## Step 1: Phase 1 and Phase 2 files (unchanged)

In [ ]:
%%writefile revcircuit.py
"""Reversible circuit model and simulator (Phase 1).

Gates are mixed-polarity multiple-control Toffoli (MPMCT) gates, written
G[P, N, q] as in Feng and Li, "A Complete Set of Transformation Rules for
Reversible Circuits", IEEE TCAD, 2026:

    P  set of positive control lines
    N  set of negative control lines
    q  target line

The gate flips q when every line in P is 1 and every line in N is 0.
Special cases: X[q] = G[{}, {}, q], CNOT[p, q] = G[{p}, {}, q], and a
Toffoli gate is G[{a, b}, {}, q].

A basis state is stored as an integer whose bit i is the value of line q_i.
A circuit is a list of gates; the leftmost gate executes first.
"""
from __future__ import annotations

import random
from dataclasses import dataclass
from itertools import product
from typing import Iterable, Iterator, List, Optional, Sequence, Tuple


# ---------------------------------------------------------------------------
# Gate
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Gate:
    """One MPMCT gate G[pos, neg, target]."""

    pos: frozenset
    neg: frozenset
    target: int

    def __post_init__(self) -> None:
        object.__setattr__(self, "pos", frozenset(self.pos))
        object.__setattr__(self, "neg", frozenset(self.neg))
        if self.pos & self.neg:
            raise ValueError("a line cannot be both a positive and a negative control")
        if self.target in self.pos or self.target in self.neg:
            raise ValueError("the target line cannot also be a control")
        if min(self.lines) < 0:
            raise ValueError("line numbers must be non-negative")
        # Bit masks used by the simulator.
        object.__setattr__(self, "_pmask", sum(1 << l for l in self.pos))
        object.__setattr__(self, "_nmask", sum(1 << l for l in self.neg))
        object.__setattr__(self, "_tmask", 1 << self.target)

    @property
    def controls(self) -> frozenset:
        return self.pos | self.neg

    @property
    def lines(self) -> frozenset:
        return self.pos | self.neg | {self.target}

    def apply(self, state: int) -> int:
        """Return the state after this gate acts on `state`."""
        if (state & self._pmask) == self._pmask and not (state & self._nmask):
            return state ^ self._tmask
        return state

    def __str__(self) -> str:
        t = self.target
        if not self.neg:
            if not self.pos:
                return f"X[q{t}]"
            if len(self.pos) == 1:
                (c,) = self.pos
                return f"CNOT[q{c},q{t}]"

        def fmt(s: frozenset) -> str:
            return "{" + ",".join(f"q{l}" for l in sorted(s)) + "}"

        return f"G[{fmt(self.pos)},{fmt(self.neg)},q{t}]"


def G(pos: Iterable[int], neg: Iterable[int], target: int) -> Gate:
    """General gate G[P, N, q]."""
    return Gate(frozenset(pos), frozenset(neg), target)


def X(q: int) -> Gate:
    """NOT gate on line q."""
    return Gate(frozenset(), frozenset(), q)


def CNOT(control: int, target: int) -> Gate:
    """Controlled-NOT (Feynman gate)."""
    return Gate(frozenset({control}), frozenset(), target)


def TOF(c1: int, c2: int, target: int) -> Gate:
    """Toffoli gate with two positive controls."""
    return Gate(frozenset({c1, c2}), frozenset(), target)


def MCT(controls: Iterable[int], target: int) -> Gate:
    """Multiple-control Toffoli gate with positive controls only."""
    return Gate(frozenset(controls), frozenset(), target)


# ---------------------------------------------------------------------------
# Circuit
# ---------------------------------------------------------------------------
class Circuit:
    """A reversible circuit on n lines: an ordered list of gates."""

    def __init__(self, n: int, gates: Iterable[Gate] = ()) -> None:
        self.n = n
        self.gates: List[Gate] = list(gates)
        for g in self.gates:
            if max(g.lines) >= n:
                raise ValueError(f"gate {g} uses a line outside q0..q{n - 1}")

    # -- container behaviour ------------------------------------------------
    def __len__(self) -> int:
        return len(self.gates)

    def __getitem__(self, i: int) -> Gate:
        return self.gates[i]

    def __iter__(self) -> Iterator[Gate]:
        return iter(self.gates)

    def __eq__(self, other: object) -> bool:
        """Structural equality: same lines and the same gate list."""
        return isinstance(other, Circuit) and self.n == other.n and self.gates == other.gates

    def __add__(self, other: "Circuit") -> "Circuit":
        """Concatenation: self executes first, then other."""
        if self.n != other.n:
            raise ValueError("circuits must have the same number of lines")
        return Circuit(self.n, self.gates + other.gates)

    def __str__(self) -> str:
        return " ".join(str(g) for g in self.gates) if self.gates else "(empty)"

    def copy(self) -> "Circuit":
        return Circuit(self.n, self.gates)

    def replace(self, i: int, j: int, new_gates: Sequence[Gate]) -> "Circuit":
        """Return a new circuit with gates[i:j] replaced by new_gates."""
        return Circuit(self.n, self.gates[:i] + list(new_gates) + self.gates[j:])

    def inverse(self) -> "Circuit":
        """Every gate is its own inverse, so the inverse is the reversed list."""
        return Circuit(self.n, reversed(self.gates))

    # -- simulation ---------------------------------------------------------
    def apply(self, state: int) -> int:
        for g in self.gates:
            state = g.apply(state)
        return state

    def permutation(self) -> Tuple[int, ...]:
        """The reversible function computed, as a tuple over all 2^n states."""
        return tuple(self.apply(s) for s in range(1 << self.n))

    def equivalent(self, other: "Circuit") -> bool:
        """Functional equivalence: both circuits compute the same function."""
        return self.n == other.n and self.permutation() == other.permutation()

    # -- display ------------------------------------------------------------
    def _bits(self, state: int) -> str:
        return " ".join(str((state >> i) & 1) for i in range(self.n))

    def truth_table(self) -> str:
        head = " ".join(f"q{i}" for i in range(self.n))
        rows = [f"{head} | {head}"]
        for s, t in enumerate(self.permutation()):
            pad = "  ".join(self._bits(s).split())
            out = "  ".join(self._bits(t).split())
            rows.append(f" {pad} |  {out}")
        return "\n".join(rows)

    def draw(self) -> str:
        """Text drawing: circled plus = target, filled dot = positive control, open dot = negative control."""
        width = len(f"q{self.n - 1}")
        rows = []
        for r in range(self.n):
            cells = []
            for g in self.gates:
                lo, hi = min(g.lines), max(g.lines)
                if r == g.target:
                    ch = "\u2295"  # target
                elif r in g.pos:
                    ch = "\u25cf"  # filled dot
                elif r in g.neg:
                    ch = "\u25cb"  # open dot
                elif lo < r < hi:
                    ch = "\u253c"  # wire crossing a gate
                else:
                    ch = "\u2500"
                cells.append("\u2500" + ch + "\u2500")
            rows.append(f"q{r}:".ljust(width + 2) + "\u2500" + "".join(cells) + "\u2500")
        return "\n".join(rows)


# ---------------------------------------------------------------------------
# Helpers for experiments
# ---------------------------------------------------------------------------
def pn_choices(lines: Sequence[int]) -> Iterator[Tuple[frozenset, frozenset]]:
    """All ways to make each given line absent, a positive or a negative control."""
    for states in product((0, 1, 2), repeat=len(lines)):
        pos = frozenset(l for l, s in zip(lines, states) if s == 1)
        neg = frozenset(l for l, s in zip(lines, states) if s == 2)
        yield pos, neg


def all_gates(n: int) -> List[Gate]:
    """Every MPMCT gate on n lines: n * 3^(n-1) gates."""
    gates = []
    for target in range(n):
        others = [l for l in range(n) if l != target]
        for pos, neg in pn_choices(others):
            gates.append(Gate(pos, neg, target))
    return gates


def random_gate(n: int, rng: Optional[random.Random] = None) -> Gate:
    rng = rng or random
    target = rng.randrange(n)
    pos, neg = set(), set()
    for l in range(n):
        if l == target:
            continue
        s = rng.randrange(3)
        if s == 1:
            pos.add(l)
        elif s == 2:
            neg.add(l)
    return Gate(frozenset(pos), frozenset(neg), target)


def random_circuit(n: int, length: int, rng: Optional[random.Random] = None) -> Circuit:
    return Circuit(n, [random_gate(n, rng) for _ in range(length)])

In [ ]:
%%writefile rules.py
"""Transformation rules for reversible circuits (Phase 1).

Rules 1 to 5 are the complete rule set RC of Feng and Li (IEEE TCAD, 2026).
Rules 6 to 10 are the rules the paper derives from RC.

Every rewrite function takes a circuit and a position i and returns a NEW
circuit with the rule applied at position i, or None when the rule does not
match there. The input circuit is never modified.

For rules stated on gates rather than positions, a `ruleK_gates` or
`ruleK_sides` helper returns the gate lists on each side of the rule.
"""
from __future__ import annotations

from itertools import combinations
from typing import Iterable, List, Optional, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, X


# ===========================================================================
# Rule 1:  A A  ==  (empty)
# ===========================================================================
def rule1_cancel(c: Circuit, i: int) -> Optional[Circuit]:
    """Delete two adjacent identical gates at positions i and i+1."""
    if 0 <= i and i + 1 < len(c) and c[i] == c[i + 1]:
        return c.replace(i, i + 2, [])
    return None


def rule1_insert(c: Circuit, i: int, gate: Gate) -> Circuit:
    """Rule 1 read right to left: insert the pair `gate gate` before position i."""
    return c.replace(i, i, [gate, gate])


# ===========================================================================
# Rule 2:  G[P, N+{p}, q] G[P+{p}, N, q]  ==  G[P, N, q]
# ===========================================================================
def rule2_gates(gate: Gate, p: int) -> List[Gate]:
    """The pair A0, A1 that Rule 2 merges into `gate`, using extra line p."""
    if p in gate.lines:
        raise ValueError("p must be a line the gate does not use")
    return [G(gate.pos, gate.neg | {p}, gate.target), G(gate.pos | {p}, gate.neg, gate.target)]


def rule2_merge(c: Circuit, i: int) -> Optional[Circuit]:
    """Merge gates i and i+1 when they are A0 A1 in the order of the paper."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    extra = a1.pos - a0.pos
    if a0.target != a1.target or len(extra) != 1:
        return None
    (p,) = extra
    merged = G(a1.pos - {p}, a1.neg, a1.target)
    if rule2_gates(merged, p) == [a0, a1]:
        return c.replace(i, i + 2, [merged])
    return None


def rule2_split(c: Circuit, i: int, p: int) -> Optional[Circuit]:
    """Rule 2 read right to left: split gate i into A0 A1 on unused line p."""
    if not (0 <= i < len(c)) or not (0 <= p < c.n) or p in c[i].lines:
        return None
    return c.replace(i, i + 1, rule2_gates(c[i], p))


# ===========================================================================
# Rule 3:  A B == B A  when a control has opposite polarity in A and B
# ===========================================================================
def rule3_condition(a: Gate, b: Gate) -> bool:
    return bool(a.pos & b.neg) or bool(b.pos & a.neg)


def rule3_commute(c: Circuit, i: int) -> Optional[Circuit]:
    """Swap gates i and i+1 when they share a control of opposite polarity."""
    if 0 <= i and i + 1 < len(c) and rule3_condition(c[i], c[i + 1]):
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# ===========================================================================
# Rule 4:  A B A C1 == C2 A B A,  A = CNOT[p,q], B = CNOT[q,p]
#          (moving a swap of p and q through a gate)
# ===========================================================================
def rule4_sides(p: int, q: int, P: Iterable[int], N: Iterable[int],
                negative: bool = False) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 4.

    negative=False is condition (a): C1 has q as positive control, target p.
    negative=True  is condition (b): C1 has q as negative control, target p.
    """
    P, N = frozenset(P), frozenset(N)
    a, b = CNOT(p, q), CNOT(q, p)
    if negative:
        c1, c2 = G(P, N | {q}, p), G(P, N | {p}, q)
    else:
        c1, c2 = G(P | {q}, N, p), G(P | {p}, N, q)
    return [a, b, a, c1], [c2, a, b, a]


def _is_cnot(g: Gate) -> bool:
    return len(g.pos) == 1 and not g.neg


def rule4_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Apply Rule 4 to the four gates starting at i, in either direction."""
    if i < 0 or i + 4 > len(c):
        return None
    seg = c.gates[i:i + 4]
    # Left to right:  A B A C1  ->  C2 A B A
    if _is_cnot(seg[0]):
        (p,) = seg[0].pos
        q = seg[0].target
        c1 = seg[3]
        if c1.target == p and q in c1.controls:
            lhs, rhs = rule4_sides(p, q, c1.pos - {q}, c1.neg - {q}, q in c1.neg)
            if seg == lhs:
                return c.replace(i, i + 4, rhs)
    # Right to left:  C2 A B A  ->  A B A C1
    if _is_cnot(seg[1]):
        (p,) = seg[1].pos
        q = seg[1].target
        c2 = seg[0]
        if c2.target == q and p in c2.controls:
            lhs, rhs = rule4_sides(p, q, c2.pos - {p}, c2.neg - {p}, p in c2.neg)
            if seg == rhs:
                return c.replace(i, i + 4, lhs)
    return None


# ===========================================================================
# Rule 5:  A0 A1 B1..Bm..B1 A1 A0  ==  B'1..B'm..B'1
#          (changes the polarity of control q0)
# ===========================================================================
def rule5_sides(P: Iterable[int], N: Iterable[int], Q: Sequence[int],
                q0: int) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 5 for control sets P, N, ordered list Q and line q0."""
    P, N, Q = frozenset(P), frozenset(N), list(Q)
    Qs = frozenset(Q)
    if not Q or len(Qs) != len(Q):
        raise ValueError("Q must be a non-empty list of distinct lines")
    a0 = G(P, N | Qs, q0)
    a1 = G(P | Qs, N, q0)
    b = [G(P | {q0} | frozenset(Q[k + 1:]), N | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    bp = [G(P | frozenset(Q[k + 1:]), N | {q0} | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    lhs = [a0, a1] + b + b[-2::-1] + [a1, a0]
    rhs = bp + bp[-2::-1]
    return lhs, rhs


def rule5_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Left to right: replace the 2m+3 gate pattern starting at i by 2m-1 gates."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    Qs = a1.pos - a0.pos
    if a0.target != a1.target or not Qs or a0.neg - a1.neg != Qs:
        return None
    if a0.pos - a1.pos or a1.neg - a0.neg:
        return None
    m = len(Qs)
    end = i + 2 * m + 3
    if end > len(c):
        return None
    order = [g.target for g in c.gates[i + 2:i + 2 + m]]
    if frozenset(order) != Qs:
        return None
    lhs, rhs = rule5_sides(a0.pos, a1.neg, order, a0.target)
    if c.gates[i:end] == lhs:
        return c.replace(i, end, rhs)
    return None


def rule5_reverse(c: Circuit, i: int, Q: Sequence[int], q0: int) -> Optional[Circuit]:
    """Right to left: expand B'1..B'm..B'1 at i, given the list Q and line q0."""
    Q = list(Q)
    end = i + 2 * len(Q) - 1
    if i < 0 or not Q or end > len(c):
        return None
    b1 = c[i]
    if b1.target != Q[0] or q0 not in b1.neg:
        return None
    try:
        lhs, rhs = rule5_sides(b1.pos - frozenset(Q[1:]), b1.neg - {q0}, Q, q0)
    except ValueError:
        return None
    if c.gates[i:end] == rhs:
        return c.replace(i, end, lhs)
    return None


# ===========================================================================
# Derived rules 6 to 10
# ===========================================================================
# Rule 6:  G[P,N,q] == product over all subsets S of Q of G[P+S, N+(Q-S), q]
def rule6_gates(gate: Gate, Q: Iterable[int]) -> List[Gate]:
    Q = sorted(set(Q))
    if set(Q) & gate.lines:
        raise ValueError("Q must not overlap the lines of the gate")
    out = []
    for r in range(len(Q) + 1):
        for sub in combinations(Q, r):
            out.append(G(gate.pos | set(sub), gate.neg | (set(Q) - set(sub)), gate.target))
    return out


def rule6_expand(c: Circuit, i: int, Q: Iterable[int]) -> Optional[Circuit]:
    """Expand gate i over all polarity patterns of the unused lines Q."""
    Q = set(Q)
    if not (0 <= i < len(c)) or not Q or Q & c[i].lines or max(Q) >= c.n or min(Q) < 0:
        return None
    return c.replace(i, i + 1, rule6_gates(c[i], Q))


# Rule 7:  two gates with the same target commute
def rule7_commute(c: Circuit, i: int) -> Optional[Circuit]:
    if 0 <= i and i + 1 < len(c) and c[i].target == c[i + 1].target:
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# Rule 8:  G[P,N,q] == X[N] G[P+N, {}, q] X[N]
def rule8_gates(gate: Gate) -> List[Gate]:
    xs = [X(l) for l in sorted(gate.neg)]
    return xs + [G(gate.pos | gate.neg, (), gate.target)] + xs


def rule8_expand(c: Circuit, i: int) -> Optional[Circuit]:
    """Replace the negative controls of gate i by X gates before and after."""
    if 0 <= i < len(c) and c[i].neg:
        return c.replace(i, i + 1, rule8_gates(c[i]))
    return None


def rule8_collapse(c: Circuit, i: int) -> Optional[Circuit]:
    """Right to left: absorb X..X B X..X starting at i into one gate."""
    k = 0
    while i + k < len(c) and not c[i + k].controls:
        k += 1
    if i < 0 or k == 0 or i + 2 * k + 1 > len(c):
        return None
    b = c[i + k]
    lines = frozenset(g.target for g in c.gates[i:i + k])
    if len(lines) != k or b.neg or not lines <= b.pos:
        return None
    merged = G(b.pos - lines, lines, b.target)
    if c.gates[i:i + 2 * k + 1] == rule8_gates(merged):
        return c.replace(i, i + 2 * k + 1, [merged])
    return None


# Rule 9:  A B1 == B2 A, where A targets p and B1, B2 differ in the polarity of p
def _rule9_pair(a: Gate, b: Gate) -> Optional[Gate]:
    """If A can pass B by Rule 9, return B with the polarity of A's target flipped."""
    p = a.target
    if p not in b.controls:
        return None
    if not (a.pos <= b.pos - {p} and a.neg <= b.neg - {p}):
        return None
    if p in b.pos:
        return G(b.pos - {p}, b.neg | {p}, b.target)
    return G(b.pos | {p}, b.neg - {p}, b.target)


def rule9_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Move a gate past its neighbour, flipping one control polarity in the neighbour."""
    if i < 0 or i + 1 >= len(c):
        return None
    g1, g2 = c[i], c[i + 1]
    flipped = _rule9_pair(g1, g2)          # A B1 -> B2 A
    if flipped is not None:
        return c.replace(i, i + 2, [flipped, g1])
    flipped = _rule9_pair(g2, g1)          # B2 A -> A B1
    if flipped is not None:
        return c.replace(i, i + 2, [g2, flipped])
    return None


# Rule 10:  A B A == B A B,  A = G[P+{p}, {}, q],  B = G[P+{q}, {}, p]
def rule10_sides(P: Iterable[int], p: int, q: int) -> Tuple[List[Gate], List[Gate]]:
    P = frozenset(P)
    a, b = G(P | {p}, (), q), G(P | {q}, (), p)
    return [a, b, a], [b, a, b]


def rule10_apply(c: Circuit, i: int) -> Optional[Circuit]:
    if i < 0 or i + 3 > len(c):
        return None
    a, b = c[i], c[i + 1]
    p, q = b.target, a.target
    if a.neg or b.neg or p == q or p not in a.pos or q not in b.pos:
        return None
    if a.pos - {p} != b.pos - {q}:
        return None
    lhs, rhs = rule10_sides(a.pos - {p}, p, q)
    if c.gates[i:i + 3] == lhs:
        return c.replace(i, i + 3, rhs)
    return None

In [ ]:
%%writefile canonical.py
"""Canonical form of a reversible circuit (Phase 2).

Implements Section IV-A of Feng and Li, "A Complete Set of Transformation
Rules for Reversible Circuits", IEEE TCAD, 2026:

  * a Hamiltonian path H = (a_0, ..., a_{2^n - 1}) of the n-hypercube,
  * the gate set Delta_H = {M_0, ..., M_{2^n - 2}}, where the n-bit gate M_i
    exchanges the neighbouring strings a_i and a_{i+1},
  * Definition 1 (canonical form) and Fact 1,
  * Algorithm 1, which builds the canonical circuit of a reversible function.

Strings follow the paper: the string "s0 s1 ... s(n-1)" gives line q_i the
value s_i. Inside the program a state is an integer whose bit i is s_i, the
same convention as revcircuit.py.

Circuits are written left to right in execution order, as in the paper. The
canonical circuit C = C_m ... C_1 C_0 therefore runs block C_m first.
"""
from __future__ import annotations

from dataclasses import dataclass
from math import factorial
from typing import Dict, List, Optional, Sequence, Tuple

from revcircuit import Circuit, G, Gate


# ---------------------------------------------------------------------------
# Strings and Hamiltonian paths
# ---------------------------------------------------------------------------
def state_of(bits: str) -> int:
    """Convert a paper-style string, e.g. "010" (q0=0, q1=1, q2=0), to a state."""
    return sum(1 << i for i, ch in enumerate(bits) if ch == "1")


def bits_of(state: int, n: int) -> str:
    """Convert a state back to a paper-style string of length n."""
    return "".join(str((state >> i) & 1) for i in range(n))


def gray_path(n: int) -> List[int]:
    """Reflected Gray code as a Hamiltonian path of the n-hypercube.

    For n = 3 this is the path of Example 4 in the paper:
    000, 001, 011, 010, 110, 111, 101, 100.
    """
    return [state_of(format(k ^ (k >> 1), f"0{n}b")) for k in range(1 << n)]


def is_hamiltonian_path(n: int, path: Sequence[int]) -> bool:
    """True when `path` visits every n-bit state once and each step flips one bit."""
    if sorted(path) != list(range(1 << n)):
        return False
    return all(bin(a ^ b).count("1") == 1 for a, b in zip(path, path[1:]))


def _resolve_path(n: int, path: Optional[Sequence[int]]) -> List[int]:
    if path is None:
        return gray_path(n)
    path = list(path)
    if not is_hamiltonian_path(n, path):
        raise ValueError("path is not a Hamiltonian path of the n-hypercube")
    return path


# ---------------------------------------------------------------------------
# The gate set Delta_H
# ---------------------------------------------------------------------------
def exchange_gate(a: int, b: int, n: int) -> Gate:
    """The n-bit gate that exchanges states a and b, which differ in one bit.

    Every other line is a control: positive where a and b hold 1, negative
    where they hold 0. All other states are left unchanged.
    """
    diff = a ^ b
    if diff == 0 or diff & (diff - 1):
        raise ValueError("the two states must differ in exactly one bit")
    target = diff.bit_length() - 1
    pos = [l for l in range(n) if l != target and (a >> l) & 1]
    neg = [l for l in range(n) if l != target and not (a >> l) & 1]
    return G(pos, neg, target)


def delta_gates(n: int, path: Optional[Sequence[int]] = None) -> List[Gate]:
    """Delta_H = [M_0, ..., M_{2^n - 2}] for the given path (Gray code by default)."""
    path = _resolve_path(n, path)
    return [exchange_gate(path[i], path[i + 1], n) for i in range(len(path) - 1)]


def circuit_from_indices(n: int, indices: Sequence[int],
                         path: Optional[Sequence[int]] = None) -> Circuit:
    """The circuit M_i1 M_i2 ... for a list of gate indices."""
    delta = delta_gates(n, path)
    return Circuit(n, [delta[i] for i in indices])


def indices_of_circuit(circuit: Circuit, path: Optional[Sequence[int]] = None) -> List[int]:
    """Indices of a circuit whose gates all belong to Delta_H."""
    lookup: Dict[Gate, int] = {g: i for i, g in enumerate(delta_gates(circuit.n, path))}
    try:
        return [lookup[g] for g in circuit]
    except KeyError as err:
        raise ValueError(f"gate {err.args[0]} is not in Delta_H") from None


# ---------------------------------------------------------------------------
# Definition 1 and Fact 1
# ---------------------------------------------------------------------------
def split_blocks(indices: Sequence[int]) -> List[List[int]]:
    """Cut an index list into maximal runs i, i+1, i+2, ... (execution order)."""
    blocks: List[List[int]] = []
    for i in indices:
        if blocks and blocks[-1][-1] + 1 == i:
            blocks[-1].append(i)
        else:
            blocks.append([i])
    return blocks


def is_canonical(indices: Sequence[int], n: int) -> bool:
    """Definition 1: blocks of consecutive gates whose first indices decrease.

    In execution order the circuit is C_m ... C_1 C_0, and the first index of
    C_i must be smaller than the first index of C_j whenever i < j.
    """
    if any(not 0 <= i <= (1 << n) - 2 for i in indices):
        return False
    starts = [block[0] for block in split_blocks(indices)]
    return all(later < earlier for earlier, later in zip(starts, starts[1:]))


def fact1_holds(indices: Sequence[int], n: int) -> bool:
    """Fact 1: at most 2^n - 1 blocks, and M_i occurs at most i + 1 times."""
    if len(split_blocks(indices)) > (1 << n) - 1:
        return False
    return all(indices.count(i) <= i + 1 for i in set(indices))


def count_canonical_forms(n: int) -> int:
    """Number of index lists that satisfy Definition 1.

    For each first index x there is either no block or a block of length
    1 .. 2^n - 1 - x, giving 2^n - x choices. The product is (2^n)!, the
    number of n-bit reversible functions.
    """
    total = 1
    for x in range((1 << n) - 1):
        total *= (1 << n) - x
    assert total == factorial(1 << n)
    return total


def max_canonical_gates(n: int) -> int:
    """Largest possible canonical circuit: every block has full length."""
    size = 1 << n
    return size * (size - 1) // 2


# ---------------------------------------------------------------------------
# Algorithm 1
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CanonicalForm:
    """Canonical circuit of a reversible function, based on a Hamiltonian path."""

    n: int
    path: Tuple[int, ...]
    blocks: Tuple[Tuple[int, ...], ...]      # execution order: C_m, ..., C_1, C_0

    @property
    def indices(self) -> List[int]:
        """Gate indices in execution order."""
        return [i for block in self.blocks for i in block]

    @property
    def circuit(self) -> Circuit:
        return circuit_from_indices(self.n, self.indices, self.path)

    def __len__(self) -> int:
        return sum(len(block) for block in self.blocks)

    def __str__(self) -> str:
        if not self.blocks:
            return "(empty)"
        return " | ".join(" ".join(f"M{i}" for i in block) for block in self.blocks)


def canonical_blocks(n: int, perm: Sequence[int], path: Sequence[int]) -> List[List[int]]:
    """Algorithm 1 on index lists. Returns the blocks in execution order.

    perm[s] is the output state for input state s. The target arrangement is
    (f(a_0), f(a_1), ...). Position i is filled by moving the wanted string
    from its position l down to i with the gates M_i M_{i+1} ... M_{l-1}.
    """
    size = 1 << n
    if sorted(perm) != list(range(size)):
        raise ValueError("perm is not a permutation of the n-bit states")
    current = list(path)
    built: List[List[int]] = []              # construction order: C_0, C_1, ...
    for i in range(size):
        wanted = perm[path[i]]
        l = current.index(wanted, i)
        if l > i:
            built.append(list(range(i, l)))
            current.insert(i, current.pop(l))
    assert current == [perm[a] for a in path]
    return built[::-1]


def canonical_from_function(n: int, perm: Sequence[int],
                            path: Optional[Sequence[int]] = None) -> CanonicalForm:
    """Canonical circuit of the reversible function perm (Algorithm 1)."""
    path = _resolve_path(n, path)
    blocks = canonical_blocks(n, perm, path)
    return CanonicalForm(n, tuple(path), tuple(tuple(b) for b in blocks))


def canonical_form(circuit: Circuit, path: Optional[Sequence[int]] = None) -> CanonicalForm:
    """Canonical form of a circuit: simulate it, then apply Algorithm 1."""
    return canonical_from_function(circuit.n, circuit.permutation(), path)


def function_from_table(n: int, table: Dict[str, str]) -> List[int]:
    """Build perm from a truth table written with paper-style strings."""
    perm = [0] * (1 << n)
    for inp, out in table.items():
        perm[state_of(inp)] = state_of(out)
    if len(table) != 1 << n or sorted(perm) != list(range(1 << n)):
        raise ValueError("the table is not a complete reversible truth table")
    return perm

## Step 2: The optimizer

One search step takes a gate, moves it along the circuit, and looks for a partner it can cancel or merge with.

| Move | Effect |
|---|---|
| Rule 1 | Two identical gates cancel |
| Rule 2 | Two gates that differ in one control polarity merge into one smaller gate |
| Rule 2 + 1 | A gate absorbs a neighbour that has one control fewer |
| Rule 5 | The paper's polarity-change pattern, when it appears literally |
| Rule 7, Rule 3, D1 | The moving gate commutes with its neighbour |
| Rule 9 | The moving gate passes; one control polarity flips |
| D2 | The moving gate passes and leaves one extra gate behind |

D1 and D2 are derived moves that the paper does not list. Step 5 checks every instance of them on 2 to 4 lines.

**Cost model.** A gate with 0 or 1 controls costs 1; with c ≥ 2 controls it costs 2^(c+1) − 3 (so 5, 13, 29, 61, ...). A gate whose controls are all negative costs one more. Pass your own `cost` function to `optimize` to change this.

In [ ]:
%%writefile optimizer.py
"""Heuristic rule-based optimizer for reversible circuits (Phase 3).

The optimizer rewrites a circuit with the transformation rules of Feng and
Li (IEEE TCAD, 2026) until the cost stops falling. One search step takes a
gate, moves it along the circuit, and looks for a partner it can cancel or
merge with.

Moves that remove gates
    Rule 1          A A                      ->  (nothing)
    Rule 2          G[P,N+p,q] G[P+p,N,q]    ->  G[P,N,q]
    Rule 2 + 1      G[P,N,q]   G[P+p,N,q]    ->  G[P,N+p,q]      ("absorb")
    Rule 5          the 2m+3 gate pattern    ->  2m-1 gates

Moves that carry a gate past its neighbour
    Rule 7          same target: the gates commute
    Rule 3          a shared control of opposite polarity: they commute
    D1 (derived)    neither target is a control of the other: they commute
    Rule 9          the gate passes; a control polarity flips in one gate
    D2 (derived)    the gate passes and leaves one extra gate behind

D1 and D2 are not listed in the paper. Both are sound (verify_phase3.py
checks every instance on 2 to 4 lines), so by the completeness theorem of
the paper they follow from Rules 1 to 5. Rule 9 is the special case of D2
in which the extra gate merges with its neighbour.

Every accepted rewrite strictly lowers the objective, so the search stops.
With check=True each rewrite is also verified by simulating the replaced
gates and the new gates on the lines they use.
"""
from __future__ import annotations

import random
from collections import Counter
from dataclasses import dataclass, field
from typing import Callable, List, Optional, Sequence, Tuple

from revcircuit import Circuit, G, Gate
import rules as R

CostFn = Callable[[Gate], int]


# ---------------------------------------------------------------------------
# Cost model
# ---------------------------------------------------------------------------
def quantum_cost(gate: Gate) -> int:
    """Quantum cost of one mixed-polarity multiple-control Toffoli gate.

    Controls:  0   1   2   3    4    5   ...   c
    Cost:      1   1   5   13   29   61  ...   2^(c+1) - 3

    This is the count of elementary gates in the decomposition of Barenco et
    al. (1995) without ancilla lines. Negative controls are free, except that
    a gate whose controls are ALL negative costs one more. The model is a
    stated assumption of this project; pass another function to change it.
    """
    c = len(gate.pos) + len(gate.neg)
    cost = 1 if c <= 1 else (1 << (c + 1)) - 3
    if c >= 1 and not gate.pos:
        cost += 1
    return cost


def circuit_cost(circuit, cost: CostFn = quantum_cost) -> int:
    """Total quantum cost of a circuit or a list of gates."""
    return sum(cost(g) for g in circuit)


# ---------------------------------------------------------------------------
# Elementary moves
# ---------------------------------------------------------------------------
def reduce_pair(a: Gate, b: Gate) -> Optional[Tuple[List[Gate], str]]:
    """Replace two gates with the same target by at most one gate, if possible."""
    if a.target != b.target:
        return None
    if a == b:
        return [], "Rule 1"
    la, lb = a.controls, b.controls
    if la == lb:
        differ = a.pos ^ b.pos                      # lines with opposite polarity
        if len(differ) == 1:
            (p,) = differ
            return [G(a.pos - {p}, a.neg - {p}, a.target)], "Rule 2"
    elif len(la ^ lb) == 1:
        big, small = (a, b) if len(la) > len(lb) else (b, a)
        (p,) = big.controls - small.controls
        if big.pos - {p} == small.pos and big.neg - {p} == small.neg:
            return [_flip(big, p)], "Rule 2 + 1"
    return None


def _flip(gate: Gate, p: int) -> Gate:
    """The same gate with the polarity of control p reversed."""
    if p in gate.pos:
        return G(gate.pos - {p}, gate.neg | {p}, gate.target)
    return G(gate.pos | {p}, gate.neg - {p}, gate.target)


def commute_tag(a: Gate, b: Gate) -> Optional[str]:
    """Name of the rule by which two adjacent gates commute, or None."""
    if a.target == b.target:
        return "Rule 7"
    if (a.pos & b.neg) or (a.neg & b.pos):
        return "Rule 3"
    if a.target not in b.controls and b.target not in a.controls:
        return "D1"
    return None


def induced_gate(a: Gate, b: Gate) -> Optional[Gate]:
    """The extra gate of rule D2.

    A = G[P1,N1,p] writes line p, and B = G[..p.., q] reads it. Then
        A B  ==  B B'' A      and      B A  ==  A B'' B
    with B'' = G[P1 + P2, N1 + N2, q], where P2, N2 are the controls of B
    other than p. Returns None when the rule does not apply.
    """
    p, q = a.target, b.target
    if p not in b.controls or q in a.controls:
        return None
    pos, neg = a.pos | (b.pos - {p}), a.neg | (b.neg - {p})
    if pos & neg:
        return None                                  # then Rule 3 applies instead
    return G(pos, neg, q)


def pass_gate(moving: Gate, fixed: Gate) -> Optional[Tuple[List[Gate], Gate, str, int]]:
    """Carry `moving` from the left of `fixed` to its right.

    Returns (gates left behind, the moving gate afterwards, rule, extra gates)
    or None when neither Rule 9 nor D2 applies.
    """
    extra = induced_gate(moving, fixed)              # moving writes, fixed reads
    if extra is not None:
        merged = reduce_pair(fixed, extra)
        if merged is not None:
            return merged[0], moving, "Rule 9", 0
        return [fixed, extra], moving, "D2", 1
    extra = induced_gate(fixed, moving)              # fixed writes, moving reads
    if extra is not None:
        merged = reduce_pair(extra, moving)
        if merged is not None:
            return [fixed], merged[0][0], "Rule 9", 0
        return [fixed, extra], moving, "D2", 1
    return None


# ---------------------------------------------------------------------------
# Checking one rewrite
# ---------------------------------------------------------------------------
def same_function(old: Sequence[Gate], new: Sequence[Gate],
                  rng: Optional[random.Random] = None, limit: int = 12) -> bool:
    """True when two gate lists act identically on the lines they use.

    Exhaustive for up to `limit` lines; beyond that, 4096 random states.
    """
    lines = sorted(set().union(*(g.lines for g in list(old) + list(new)))) if (old or new) else []
    index = {l: i for i, l in enumerate(lines)}

    def remap(gates):
        return [G([index[l] for l in g.pos], [index[l] for l in g.neg], index[g.target])
                for g in gates]

    a, b = remap(old), remap(new)
    k = len(lines)
    if k <= limit:
        states = range(1 << k)
    else:
        rng = rng or random.Random(0)
        states = [rng.getrandbits(k) for _ in range(4096)]
    for s in states:
        x = y = s
        for g in a:
            x = g.apply(x)
        for g in b:
            y = g.apply(y)
        if x != y:
            return False
    return True


# ---------------------------------------------------------------------------
# The optimizer
# ---------------------------------------------------------------------------
@dataclass
class OptimizationResult:
    original: Circuit
    circuit: Circuit
    rule_counts: Counter = field(default_factory=Counter)
    rewrites: int = 0
    passes: int = 0
    cost_fn: CostFn = quantum_cost

    @property
    def gates_before(self) -> int:
        return len(self.original)

    @property
    def gates_after(self) -> int:
        return len(self.circuit)

    @property
    def cost_before(self) -> int:
        return circuit_cost(self.original, self.cost_fn)

    @property
    def cost_after(self) -> int:
        return circuit_cost(self.circuit, self.cost_fn)

    def summary(self) -> str:
        def change(before, after):
            pct = 100.0 * (before - after) / before if before else 0.0
            return f"{before} -> {after} ({pct:.1f}% lower)"

        lines = [f"Gate count:   {change(self.gates_before, self.gates_after)}",
                 f"Quantum cost: {change(self.cost_before, self.cost_after)}",
                 f"Rewrites accepted: {self.rewrites} in {self.passes} passes"]
        if self.rule_counts:
            lines.append("Rule usage: " + ", ".join(
                f"{name} x{count}" for name, count in sorted(self.rule_counts.items())))
        return "\n".join(lines)


class _Search:
    def __init__(self, objective: str, window: int, extra_gates: int,
                 cost: CostFn, check: bool) -> None:
        if objective not in ("cost", "gates"):
            raise ValueError('objective must be "cost" or "gates"')
        self.objective, self.window, self.extra_gates = objective, window, extra_gates
        self.cost, self.check = cost, check
        self.counts: Counter = Counter()
        self.rewrites = 0
        self.rng = random.Random(0)

    def score(self, gates: Sequence[Gate]) -> Tuple[int, int]:
        qc, count = sum(self.cost(g) for g in gates), len(gates)
        return (qc, count) if self.objective == "cost" else (count, qc)

    def walk(self, gates: List[Gate], i: int):
        """Move gate i to the right; return the best improving rewrite found."""
        moving, left, tags, growth = gates[i], [], [], 0
        best = None
        for k in range(i + 1, min(len(gates), i + 1 + self.window)):
            fixed = gates[k]
            reduced = reduce_pair(moving, fixed)
            if reduced is not None:
                new = left + reduced[0]
                gain = tuple(o - n for o, n in zip(self.score(gates[i:k + 1]), self.score(new)))
                if gain > (0, 0) and (best is None or gain > best[0]):
                    best = (gain, k, new, tags + [reduced[1]])
            tag = commute_tag(moving, fixed)
            if tag is not None:
                left, tags = left + [fixed], tags + [tag]
                continue
            passed = pass_gate(moving, fixed)
            if passed is None or growth + passed[3] > self.extra_gates:
                break
            left, moving = left + passed[0], passed[1]
            tags, growth = tags + [passed[2]], growth + passed[3]
        return best

    def sweep(self, gates: List[Gate]) -> bool:
        """One left-to-right sweep. Returns True when the circuit changed."""
        changed, i = False, 0
        while i < len(gates):
            best = self.walk(gates, i)
            if best is None:
                i += 1
                continue
            _, k, new, tags = best
            if self.check and not same_function(gates[i:k + 1], new, self.rng):
                raise AssertionError("optimizer produced a non-equivalent rewrite")
            gates[i:k + 1] = new
            self.counts.update(tags)
            self.rewrites += 1
            changed = True
        return changed

    def rule5(self, n: int, gates: List[Gate]) -> bool:
        """Apply Rule 5 wherever its pattern appears literally."""
        changed, i = False, 0
        while i < len(gates):
            new = R.rule5_apply(Circuit(n, gates), i)
            if new is not None and self.score(new.gates) < self.score(gates):
                gates[:] = new.gates
                self.counts["Rule 5"] += 1
                self.rewrites += 1
                changed = True
            else:
                i += 1
        return changed


def optimize(circuit: Circuit, objective: str = "cost", window: int = 50,
             extra_gates: int = 1, cost: CostFn = quantum_cost,
             check: bool = True, max_passes: int = 100) -> OptimizationResult:
    """Reduce a reversible circuit with the transformation rules.

    objective    "cost" minimises quantum cost, then gate count;
                 "gates" minimises gate count, then quantum cost.
    window       how many gates a moving gate may travel past.
    extra_gates  how many extra gates (rule D2) one move may leave behind.
    check        verify every rewrite by simulation on the lines it touches.
    """
    search = _Search(objective, window, extra_gates, cost, check)
    gates = list(circuit.gates)
    passes = 0
    while passes < max_passes:
        passes += 1
        changed = search.sweep(gates)
        # The same search on the reversed circuit moves gates to the left.
        # Each gate is its own inverse, so the reversed circuit is the
        # inverse circuit and every rule stays valid.
        gates.reverse()
        changed |= search.sweep(gates)
        gates.reverse()
        changed |= search.rule5(circuit.n, gates)
        if not changed:
            break
    return OptimizationResult(circuit, Circuit(circuit.n, gates), search.counts,
                              search.rewrites, passes, cost)

## Step 3: Exact minimum circuits for 3 lines

In [ ]:
%%writefile exact.py
"""Exact minimum circuits for small reversible functions (Phase 3).

For n lines there are (2^n)! reversible functions. For n = 3 that is 40320,
few enough to search completely. The search uses every mixed-polarity
multiple-control Toffoli gate on n lines as its library (27 gates for n = 3).

    minimum gate count    breadth-first search
    minimum quantum cost  Dijkstra's algorithm with the project cost model

The tables give the true optimum for every 3-line function, so the
heuristic optimizer can be measured against it.
"""
from __future__ import annotations

import heapq
from typing import Callable, Dict, List, Optional, Tuple

from revcircuit import Circuit, Gate, all_gates
from optimizer import quantum_cost

Perm = Tuple[int, ...]


class ExactTable:
    """Minimum gate count and minimum quantum cost of every n-line function."""

    def __init__(self, n: int = 3, cost: Callable[[Gate], int] = quantum_cost) -> None:
        if n > 3:
            raise ValueError("complete search is practical only for n <= 3")
        self.n = n
        self.gates: List[Gate] = all_gates(n)
        size = 1 << n
        self._acts = [tuple(g.apply(s) for s in range(size)) for g in self.gates]
        self._costs = [cost(g) for g in self.gates]
        self._identity: Perm = tuple(range(size))
        self.min_gates, self._gate_parent = self._breadth_first()
        self.min_cost, self._cost_parent = self._dijkstra()

    def _breadth_first(self):
        dist: Dict[Perm, int] = {self._identity: 0}
        parent: Dict[Perm, Optional[Tuple[Perm, int]]] = {self._identity: None}
        frontier = [self._identity]
        while frontier:
            nxt = []
            for p in frontier:
                d = dist[p] + 1
                for gi, act in enumerate(self._acts):
                    q = tuple([act[x] for x in p])        # circuit p, then gate gi
                    if q not in dist:
                        dist[q] = d
                        parent[q] = (p, gi)
                        nxt.append(q)
            frontier = nxt
        return dist, parent

    def _dijkstra(self):
        dist: Dict[Perm, int] = {self._identity: 0}
        parent: Dict[Perm, Optional[Tuple[Perm, int]]] = {self._identity: None}
        done = set()
        heap = [(0, self._identity)]
        while heap:
            d, p = heapq.heappop(heap)
            if p in done:
                continue
            done.add(p)
            for gi, act in enumerate(self._acts):
                q = tuple([act[x] for x in p])
                nd = d + self._costs[gi]
                if nd < dist.get(q, 1 << 60):
                    dist[q] = nd
                    parent[q] = (p, gi)
                    heapq.heappush(heap, (nd, q))
        return dist, parent

    def _rebuild(self, perm: Perm, parent) -> Circuit:
        gates = []
        while parent[perm] is not None:
            perm, gi = parent[perm]
            gates.append(self.gates[gi])
        return Circuit(self.n, gates[::-1])

    def min_gate_circuit(self, perm) -> Circuit:
        """A circuit with the fewest gates for the function perm."""
        return self._rebuild(tuple(perm), self._gate_parent)

    def min_cost_circuit(self, perm) -> Circuit:
        """A circuit with the lowest quantum cost for the function perm."""
        return self._rebuild(tuple(perm), self._cost_parent)

## Step 4: Verification script

In [ ]:
%%writefile verify_phase3.py
"""Phase 3 verification: the rule-based optimizer.

  A. The two derived moves D1 and D2 are sound on every instance (2 to 4
     lines), and the three commutation tests find every commuting gate pair.
  B. The cost model.
  C. Known examples reduce to the expected circuits.
  D. Random circuits: the optimized circuit always computes the same
     function and never costs more.
  E. Redundancy recovery: redundancy is injected into a circuit with the
     Phase 1 rules, and the optimizer has to remove it again.
  F. Gap to the exact optimum for 3-line functions.
  G. Which rules fired.

Run:  python verify_phase3.py
"""
from __future__ import annotations

import random
import sys
import time
from collections import Counter

from revcircuit import CNOT, Circuit, G, TOF, X, all_gates, random_circuit, random_gate
import rules as R
from canonical import canonical_form, canonical_from_function
from optimizer import (circuit_cost, commute_tag, optimize, pass_gate,
                       quantum_cost)
from exact import ExactTable

# The circuit drawings use Unicode symbols; make sure they print on Windows too.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

rng = random.Random(2026)
failures = 0
usage: Counter = Counter()


def report(label: str, ok: bool, detail: str = "") -> None:
    global failures
    failures += not ok
    print(f"  {label:<52}{'PASS' if ok else 'FAIL'}" + (f"  {detail}" if detail else ""))


def run(circuit: Circuit, **kw):
    """Optimize, record rule usage, and confirm the function is unchanged."""
    result = optimize(circuit, **kw)
    usage.update(result.rule_counts)
    return result


# ---------------------------------------------------------------------------
# A. Derived moves
# ---------------------------------------------------------------------------
def part_a() -> None:
    print("A. Derived moves D1 and D2, every gate pair")
    print(f"  {'Lines':<7}{'Pairs':>8}{'Commuting':>11}{'Found':>8}"
          f"{'Rule 9':>8}{'D2':>7}  Result")
    for n in (2, 3, 4):
        gates = all_gates(n)
        commuting = found = rule9 = d2 = 0
        ok = True
        for a in gates:
            for b in gates:
                ab = Circuit(n, [a, b])
                really = ab.equivalent(Circuit(n, [b, a]))
                tag = commute_tag(a, b)
                ok &= really or tag is None           # a tag must never be wrong
                if a != b and really:
                    commuting += 1
                    found += tag is not None
                if tag is None:
                    passed = pass_gate(a, b)
                    if passed is not None:
                        left, moving, name, extra = passed
                        ok &= Circuit(n, left + [moving]).equivalent(ab)
                        ok &= len(left) + 1 == 2 + extra
                        rule9 += name == "Rule 9"
                        d2 += name == "D2"
        ok &= commuting == found
        global failures
        failures += not ok
        print(f"  {n:<7}{len(gates) ** 2:>8}{commuting:>11}{found:>8}{rule9:>8}{d2:>7}  "
              f"{'PASS' if ok else 'FAIL'}")
    print("  'Found' = commuting pairs recognised by Rule 7, Rule 3 or D1.\n")


# ---------------------------------------------------------------------------
# B. Cost model
# ---------------------------------------------------------------------------
def part_b() -> None:
    print("B. Cost model")
    table = [quantum_cost(G(range(1, c + 1), [], 0)) for c in range(6)]
    report("Positive controls 0..5 cost 1, 1, 5, 13, 29, 61", table == [1, 1, 5, 13, 29, 61])
    report("One negative control among two costs 5", quantum_cost(G([1], [2], 0)) == 5)
    report("All controls negative costs one more",
           quantum_cost(G([], [1], 0)) == 2 and quantum_cost(G([], [1, 2], 0)) == 6)
    print()


# ---------------------------------------------------------------------------
# C. Known examples
# ---------------------------------------------------------------------------
def part_c() -> None:
    print("C. Known examples")
    lhs5, rhs5 = R.rule5_sides([], [], [1, 2], 0)
    cases = [
        ("Paper Example 2: X CNOT X",
         Circuit(2, [X(0), CNOT(0, 1), X(0)]), Circuit(2, [G([], [0], 1)])),
        ("Three CNOT gates to two",
         Circuit(3, [CNOT(1, 2), CNOT(0, 1), CNOT(1, 2)]), Circuit(3, [CNOT(0, 1), CNOT(0, 2)])),
        ("X Toffoli X to a negative control",
         Circuit(3, [X(0), TOF(0, 1, 2), X(0)]), Circuit(3, [G([1], [0], 2)])),
        ("CNOT and Toffoli on one target",
         Circuit(3, [CNOT(0, 2), TOF(0, 1, 2)]), Circuit(3, [G([0], [1], 2)])),
        ("Canonical circuit of CNOT[q0,q1] (6 gates)",
         canonical_form(Circuit(3, [CNOT(0, 1)])).circuit, Circuit(3, [CNOT(0, 1)])),
        ("Canonical circuit of paper Example 7 (6 gates)",
         canonical_form(Circuit(3, [CNOT(0, 1), CNOT(1, 2)])).circuit,
         Circuit(3, [CNOT(0, 1), CNOT(1, 2)])),
        ("Rule 5 pattern, m = 2 (7 gates to 3)", Circuit(3, lhs5), Circuit(3, rhs5)),
    ]
    for label, start, expected in cases:
        result = run(start)
        ok = (result.circuit.equivalent(start)
              and len(result.circuit) <= len(expected)
              and result.cost_after <= circuit_cost(expected))
        report(label, ok, f"{len(start)} -> {len(result.circuit)} gates, "
                          f"cost {result.cost_before} -> {result.cost_after}")
    start = Circuit(3, [X(0), TOF(0, 1, 2), CNOT(1, 2), X(0), CNOT(1, 2)])
    result = run(start)
    print("\n  Example run:")
    for line in start.draw().splitlines():
        print("    " + line)
    print("    becomes")
    for line in result.circuit.draw().splitlines():
        print("    " + line)
    for line in result.summary().splitlines():
        print("    " + line)
    print()


# ---------------------------------------------------------------------------
# D. Random circuits
# ---------------------------------------------------------------------------
def part_d() -> None:
    print("D. Random circuits: same function, never a higher cost")
    print(f"  {'Lines':<7}{'Circuits':>9}{'Gates before':>14}{'after':>8}"
          f"{'Cost before':>13}{'after':>9}  Result")
    for n, count in ((3, 150), (4, 150), (5, 100), (6, 100), (7, 60), (8, 40)):
        g0 = g1 = c0 = c1 = 0
        ok = True
        for k in range(count):
            circuit = random_circuit(n, rng.randint(5, 40), rng)
            objective = "cost" if k % 2 == 0 else "gates"
            result = run(circuit, objective=objective)
            ok &= result.circuit.equivalent(circuit)
            if objective == "cost":
                ok &= result.cost_after <= result.cost_before
            else:
                ok &= result.gates_after <= result.gates_before
            g0 += result.gates_before
            g1 += result.gates_after
            c0 += result.cost_before
            c1 += result.cost_after
        global failures
        failures += not ok
        print(f"  {n:<7}{count:>9}{g0:>14}{g1:>8}{c0:>13}{c1:>9}  {'PASS' if ok else 'FAIL'}")
    print("  Equivalence is checked by simulating all 2^n inputs.\n")


# ---------------------------------------------------------------------------
# E. Redundancy recovery
# ---------------------------------------------------------------------------
def random_nct(n: int, length: int) -> Circuit:
    """Random circuit of X, CNOT and Toffoli gates, the gate set adders are built from."""
    gates = []
    for _ in range(length):
        kind = rng.choice((0, 1, 1, 2, 2))
        lines = rng.sample(range(n), kind + 1)
        gates.append(G(lines[1:], [], lines[0]))
    return Circuit(n, gates)


def scramble(c: Circuit, steps: int, max_len: int = 80) -> Circuit:
    """Rewrite a circuit with randomly chosen Phase 1 rules. The function is unchanged."""
    for _ in range(steps):
        i = rng.randrange(len(c) + 1)
        move = rng.randrange(12)
        new = None
        if move == 0 and len(c) + 2 <= max_len:
            new = R.rule1_insert(c, i, random_gate(c.n, rng))
        elif move == 1 and len(c) + 1 <= max_len:
            new = R.rule2_split(c, i, rng.randrange(c.n))
        elif move == 2 and len(c) + 2 * c.n <= max_len:
            new = R.rule8_expand(c, i)
        else:
            fn = (R.rule1_cancel, R.rule2_merge, R.rule3_commute, R.rule4_apply,
                  R.rule5_apply, R.rule7_commute, R.rule8_collapse, R.rule9_apply,
                  R.rule10_apply)[move - 3]
            new = fn(c, i)
        if new is not None:
            c = new
    return c


def part_e() -> None:
    print("E. Redundancy recovery (X, CNOT, Toffoli circuits of 6 to 12 gates)")
    print(f"  {'Lines':<7}{'Circuits':>9} | {'Average gates':^31} | {'Average quantum cost':^31} |"
          f"{'Back to original':>18}")
    print(f"  {'':<7}{'':>9} | {'original':>9}{'padded':>10}{'optimized':>11} |"
          f" {'original':>9}{'padded':>10}{'optimized':>11} |{'cost or lower':>18}")
    for n, count in ((4, 150), (5, 150), (6, 100)):
        sums = [0] * 6
        back = 0
        ok = True
        for _ in range(count):
            original = random_nct(n, rng.randint(6, 12))
            padded = scramble(original, 80)
            result = run(padded)
            ok &= result.circuit.equivalent(original)
            values = (len(original), len(padded), len(result.circuit),
                      circuit_cost(original), circuit_cost(padded), result.cost_after)
            sums = [s + v for s, v in zip(sums, values)]
            back += result.cost_after <= circuit_cost(original)
        global failures
        failures += not ok
        a = [s / count for s in sums]
        print(f"  {n:<7}{count:>9} | {a[0]:>9.1f}{a[1]:>10.1f}{a[2]:>11.1f} |"
              f" {a[3]:>9.1f}{a[4]:>10.1f}{a[5]:>11.1f} |{100 * back / count:>17.1f}%"
              f"  {'PASS' if ok else 'FAIL'}")
    print("  'Padded' = the original after 80 random rule steps that add redundancy.")
    print("  PASS means every optimized circuit still computes the original function.\n")


# ---------------------------------------------------------------------------
# F. Gap to the exact optimum on 3 lines
# ---------------------------------------------------------------------------
def part_f() -> None:
    print("F. Gap to the exact optimum, 3-line functions")
    start = time.time()
    table = ExactTable(3)
    sizes = Counter(table.min_gates.values())
    print(f"  Exact tables for all {len(table.min_gates)} functions built in "
          f"{time.time() - start:.1f} s.")
    print("  Functions by minimum gate count: "
          + ", ".join(f"{k} gates: {sizes[k]}" for k in sorted(sizes)))
    print(f"  Highest minimum quantum cost: {max(table.min_cost.values())}\n")
    samples = 1000
    print(f"  {samples} random functions per row.")
    print(f"  {'Starting circuit':<28}{'Measure':<10}{'Start':>8}{'Optimized':>11}"
          f"{'Optimum':>9}{'Reached optimum':>17}")
    ok = True
    for start_kind in ("canonical form", "padded optimal circuit"):
        for objective, measure in (("gates", "gates"), ("cost", "cost")):
            s0 = s1 = so = hit = 0
            for _ in range(samples):
                perm = tuple(rng.sample(range(8), 8))
                if start_kind == "canonical form":
                    circuit = canonical_from_function(3, perm).circuit
                elif objective == "gates":
                    circuit = scramble(table.min_gate_circuit(perm), 60)
                else:
                    circuit = scramble(table.min_cost_circuit(perm), 60)
                result = run(circuit, objective=objective)
                ok &= result.circuit.permutation() == perm
                if objective == "gates":
                    before, after, best = len(circuit), len(result.circuit), table.min_gates[perm]
                else:
                    before, after, best = result.cost_before, result.cost_after, table.min_cost[perm]
                ok &= after >= best                    # nothing can beat the optimum
                s0, s1, so, hit = s0 + before, s1 + after, so + best, hit + (after == best)
            print(f"  {start_kind:<28}{measure:<10}{s0 / samples:>8.2f}{s1 / samples:>11.2f}"
                  f"{so / samples:>9.2f}{100 * hit / samples:>16.1f}%")
    print()
    report("Results consistent with the exact tables", ok)
    print()


def part_g() -> None:
    print("G. Rule usage over all experiments above")
    total = sum(usage.values())
    for name, count in usage.most_common():
        print(f"  {name:<12}{count:>8}  {100 * count / total:5.1f}%")
    print()


def main() -> int:
    start = time.time()
    print("Phase 3 verification: rule-based optimizer\n")
    part_a()
    part_b()
    part_c()
    part_d()
    part_e()
    part_f()
    part_g()
    print(f"Time: {time.time() - start:.1f} s")
    print("\nAll checks passed." if failures == 0 else f"\n{failures} check(s) FAILED.")
    return 1 if failures else 0


if __name__ == "__main__":
    sys.exit(main())

## Step 5: Run the verification

Every check should show **PASS**, and the output should end with `All checks passed.`

In [ ]:
!python verify_phase3.py

## Step 6: Try it yourself

Edit the gate list and run the cell again. Lines are numbered from 0.

- `optimize(c)` minimises quantum cost first, then gate count.
- `optimize(c, objective="gates")` minimises gate count first.

Drawing symbols: `⊕` target, `●` positive control, `○` negative control.

In [ ]:
import importlib
import revcircuit, rules, canonical, optimizer
for m in (revcircuit, rules, canonical, optimizer):
    importlib.reload(m)

from revcircuit import Circuit, G, X, CNOT, TOF
from optimizer import optimize, circuit_cost

c = Circuit(4, [X(0), TOF(0, 1, 2), CNOT(1, 2), X(0), CNOT(1, 2),
                CNOT(2, 3), TOF(0, 1, 3), CNOT(2, 3), CNOT(0, 3)])

result = optimize(c)

print("Before:", c)
print(c.draw())
print()
print("After: ", result.circuit)
print(result.circuit.draw())
print()
print(result.summary())
print("Same function as before?", result.circuit.equivalent(c))